In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 09 · Inferencia — del modelo registrado a una función que cualquiera puede usar

> **Esqueleto.** Guía completa: `curso/kit-estudiante/guia-validacion-registro-y-app.md`, secciones 6 a 8.

Un modelo entrenado no es un producto. Un producto recibe datos **sucios o incompletos de una persona real** y
responde con algo útil o con un **error claro**. Esta etapa construye ese puente:

1. **Exportar** el modelo `champion` a un archivo que viaja con la app.
2. Escribir el **esquema de entrada**: qué columnas, qué rangos son posibles, qué categorías.
3. Probar la función de predicción con datos buenos, incompletos y **imposibles**.
4. Comprobar que el archivo exportado **es** el modelo que se evaluó.
5. Completar la **ficha del modelo**.

## 0. Datos y rutas

In [ ]:
import json

import joblib
import pandas as pd

raiz = Path.cwd().resolve().parent
train_path = raiz / "data" / "processed" / "train.joblib"
train = joblib.load(train_path) if train_path.exists() else None
if train is None:
    print("Todavía no hay datos de entrenamiento: completa y ejecuta antes el notebook 03_preprocesamiento.")

# === Configuración: completa estas líneas ===
TARGET = None             # TODO: tu columna objetivo
NUMERICAS = None          # TODO: lista de columnas numéricas CRUDAS que el modelo necesita
CATEGORICAS = None        # TODO: lista de columnas categóricas CRUDAS que el modelo necesita
CLASES = None             # TODO: nombres de las clases, en el orden de predict_proba (p. ej. ["no", "si"])
MATRIZ_DE_COSTOS = None   # TODO: opcional, matriz k x k (filas = clase real, columnas = clase decidida); None si decides por la más probable
RANGOS_POSIBLES = {}      # TODO: {"columna": (mínimo, máximo)} de lo que es POSIBLE en el mundo real (edad: 0 a 120, no 14 a 56)

configurado = train is not None and all(x is not None for x in (NUMERICAS, CATEGORICAS, CLASES))
if not configurado:
    print("Completa NUMERICAS, CATEGORICAS y CLASES para continuar.")

## 1. Exportar el modelo `champion`

La fuente de verdad es el **registro**. Si no hay conexión, se usa la copia local de la etapa 7 y el notebook lo dice,
para que nadie crea que usa una cosa y use otra.

In [ ]:
modelo = None
try:
    import mlflow
    from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name

    registro = raiz / "reports" / "08_registro.json"
    if registro.exists():
        info_registro = json.loads(registro.read_text(encoding="utf-8"))
        with warnings.catch_warnings(record=True):
            warnings.simplefilter("always")
            donde = configure_mlflow()
        modelo = mlflow.sklearn.load_model(f"models:/{info_registro['modelo_registrado']}@champion")
        origen = f"registro de MLflow ({donde}), {info_registro['modelo_registrado']}@champion"
except Exception as error:
    print("No se pudo leer el registro:", str(error).splitlines()[0][:90])

if modelo is None and (raiz / "models" / "final_pipeline.joblib").exists():
    modelo = joblib.load(raiz / "models" / "final_pipeline.joblib")
    origen = "copia local de la etapa 7 (el registro no respondió)"

if modelo is None:
    print("Pendiente: termina los notebooks 07 y 08.")
else:
    (raiz / "models").mkdir(exist_ok=True)
    joblib.dump(modelo, raiz / "models" / "model.joblib")
    print("modelo exportado a models/model.joblib desde:", origen)

## 2. El esquema de entrada

La app y la función de predicción necesitan saber qué se puede preguntar: qué columnas, qué rango es **posible** y
qué categorías conoce el modelo. Se calcula una vez con el entrenamiento y se guarda como un archivo que viaja con el
modelo (`models/input_schema.json`).

> `RANGOS_POSIBLES` es **conocimiento tuyo**, no de los datos: una edad de 200 es imposible aunque nunca haya
> aparecido. Fuera de ese rango, la predicción **se niega** a responder.

In [ ]:
from proyecto.inference.schema import build_schema

schema = None
if configurado:
    schema = build_schema(
        train, NUMERICAS, CATEGORICAS, CLASES,
        cost_matrix=MATRIZ_DE_COSTOS, valid_range=RANGOS_POSIBLES,
        path=raiz / "models" / "input_schema.json",
    )
    print(f"esquema guardado: {len(NUMERICAS)} numéricas, {len(CATEGORICAS)} categóricas, clases {CLASES}")
else:
    print("Pendiente: configura las columnas (sección 0).")

## 3. Probar la predicción

Tres casos que importan: una fila normal, una **incompleta** (el usuario marcó "no sé") y una con un dato
**imposible**. Qué se espera: la primera responde; la segunda también (el pipeline imputa); la tercera **se niega y
explica por qué**.

In [ ]:
from proyecto.inference.predict import InputValidationError, predict_with_scores

if modelo is not None and schema is not None:
    entrada = train[NUMERICAS + CATEGORICAS]
    normal = entrada.head(1).copy()
    print("— fila normal —")
    print(predict_with_scores(normal, modelo, schema).round(3).to_string())

    incompleta = normal.copy()
    incompleta[NUMERICAS[0]] = float("nan")
    print("\n— con un dato faltante —")
    print(predict_with_scores(incompleta, modelo, schema).round(3).to_string())

    if RANGOS_POSIBLES:
        columna = next(iter(RANGOS_POSIBLES))
        imposible = normal.copy()
        imposible[columna] = RANGOS_POSIBLES[columna][1] * 1000 + 1
        try:
            predict_with_scores(imposible, modelo, schema)
            print("\nERROR: debía negarse a predecir con un valor imposible")
        except InputValidationError as error:
            print("\n— dato imposible —\nError claro:", error)
    else:
        print("\nPendiente: define RANGOS_POSIBLES para probar un dato imposible.")
else:
    print("Pendiente: exporta el modelo (sección 1) y crea el esquema (sección 2).")

## 4. Comprobar que el archivo exportado es el modelo que se evaluó

Antes de dárselo a la app: `models/model.joblib` debe dar **las mismas métricas** que midiste en la prueba (etapa 7).
Si no coinciden, estás entregando un modelo distinto al que evaluaste.

In [ ]:
# TODO: carga la prueba (data/processed/test.joblib), calcula tu métrica principal con models/model.joblib y compárala
#       con la de reports/07_metricas_test.json. Usa assert abs(a - b) < 1e-9, "el modelo exportado no es el que se evaluó".
print("Pendiente: comprobación de que el exportado es el evaluado.")

## 5. La ficha del modelo

Un documento corto que cualquiera pueda leer antes de usar el modelo: qué es, con qué datos, qué rendimiento tiene y
**qué no hay que hacer con él**. La plantilla está en `models/ficha_modelo.md`; complétala **a mano** con tus
resultados (no con cifras que no medimos).

In [ ]:
ficha = raiz / "models" / "ficha_modelo.md"
if ficha.exists():
    texto = ficha.read_text(encoding="utf-8")
    pendientes = texto.count("_______________")
    print(f"ficha_modelo.md: {pendientes} campos en blanco" if pendientes else "ficha_modelo.md completa")
else:
    print("No existe models/ficha_modelo.md (viene en la plantilla).")

## 6. Siguiente: la app

Con `models/model.joblib` y `models/input_schema.json` en su sitio, la app de la plantilla ya funciona:

```bash
streamlit run app/streamlit_app.py
```

Comprueba que: el formulario aparece, una predicción normal responde, **"No sé" no rompe nada** y un dato imposible
muestra un error claro. El despliegue en internet es la clase 5.